# Tema 4 código

**Modelos modernos con Transformers**

Notebook construido con el criterio solicitado: se conserva el **código exacto del PDF** y se agregan complementos mínimos para que el alumno pueda ejecutarlo, comparar resultados y visualizar salidas.


## 0. Instalación de librerías

Ejecuta esta celda solo si tu entorno no tiene instaladas las librerías necesarias.

In [ ]:
# Complemento mínimo para ejecución
# Descomenta si trabajas en Google Colab o si tu entorno no tiene estas librerías.
# !pip install transformers torch matplotlib pandas -q

## 1. Código 1 del PDF: tokenización con Hugging Face

Este bloque conserva el código del PDF. Carga un tokenizador preentrenado basado en BERT y transforma una oración en tensores que pueden ser procesados por el modelo.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
texto = "The movie was fantastic and emotionally engaging."
tokens = tokenizer(texto, return_tensors="pt")
print(tokens)

### Complemento: visualización de tokens

Esta celda no sustituye el código del PDF; solo muestra de forma más clara cómo el texto se convierte en tokens.

In [ ]:
# Complemento mínimo: inspección didáctica de la tokenización
tokens_texto = tokenizer.tokenize(texto)
ids = tokenizer.convert_tokens_to_ids(tokens_texto)

for token, token_id in zip(tokens_texto, ids):
    print(f"{token:15s} -> {token_id}")

In [ ]:
# Complemento mínimo: visualización simple de IDs de tokens
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.bar(tokens_texto, ids)
plt.xticks(rotation=45, ha="right")
plt.title("IDs generados por el tokenizador BERT")
plt.xlabel("Token")
plt.ylabel("ID del token")
plt.tight_layout()
plt.show()

## 2. Código 2 del PDF: análisis de sentimientos usando BERT

El PDF presenta este flujo con `pipeline` de Hugging Face.

In [ ]:
from transformers import pipeline

classifier = pipeline(
 "sentiment-analysis",
 model="bert-base-uncased"
)
resultado = classifier("The product works perfectly and exceeded expectations.")
print(resultado)

### Complemento técnico para ejecución estable

En algunos entornos, `bert-base-uncased` puede cargarse sin una cabeza de clasificación ajustada para sentimiento. Por eso se agrega una versión funcional de respaldo con un modelo ya afinado para análisis de sentimientos.

In [ ]:
# Complemento mínimo: versión estable si se requiere una cabeza de clasificación afinada
from transformers import pipeline

try:
    classifier_bert_estable = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )
    resultado_bert_estable = classifier_bert_estable("The product works perfectly and exceeded expectations.")
    print(resultado_bert_estable)
except Exception as e:
    print("No fue posible cargar el modelo desde Hugging Face en este entorno.")
    print("Detalle:", e)

## 3. Código 3 del PDF: análisis de sentimientos usando RoBERTa

In [ ]:
from transformers import pipeline

classifier = pipeline(
 "sentiment-analysis",
 model="cardiffnlp/twitter-roberta-base-sentiment"
)
texto = "The service was fast and the experience was excellent."
print(classifier(texto))

### Complemento: interpretación de etiquetas de RoBERTa

El modelo `cardiffnlp/twitter-roberta-base-sentiment` suele devolver etiquetas como `LABEL_0`, `LABEL_1` y `LABEL_2`. Esta celda ayuda al alumno a interpretarlas.

In [ ]:
# Complemento mínimo: mapeo habitual para cardiffnlp/twitter-roberta-base-sentiment
mapa_roberta = {
    "LABEL_0": "negativo",
    "LABEL_1": "neutral",
    "LABEL_2": "positivo"
}

try:
    resultado_roberta = classifier(texto)[0]
    etiqueta_original = resultado_roberta["label"]
    print("Texto:", texto)
    print("Etiqueta original:", etiqueta_original)
    print("Interpretación:", mapa_roberta.get(etiqueta_original, etiqueta_original))
    print("Score:", resultado_roberta["score"])
except Exception as e:
    print("No fue posible ejecutar el modelo RoBERTa en este entorno.")
    print("Detalle:", e)

## 4. Código 4 del PDF: análisis de sentimientos usando DistilBERT

In [ ]:
from transformers import pipeline

classifier = pipeline(
 "sentiment-analysis",
 model="distilbert-base-uncased-finetuned-sst-2-english"
)
texto = "The update made the application slower."
print(classifier(texto))

## 5. Complemento integrador: comparación visual de predicciones

Esta sección usa los mismos modelos del PDF para comparar predicciones en varias frases. Es un complemento didáctico con visualización.

In [ ]:
# Complemento mínimo: comparación de modelos y visualización
import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

textos_prueba = [
    "The product works perfectly and exceeded expectations.",
    "The service was fast and the experience was excellent.",
    "The update made the application slower.",
    "The app is useful, but the latest version has some issues."
]

modelos = {
    "BERT_PDF": "bert-base-uncased",
    "RoBERTa_PDF": "cardiffnlp/twitter-roberta-base-sentiment",
    "DistilBERT_PDF": "distilbert-base-uncased-finetuned-sst-2-english"
}

resultados = []

for nombre, modelo in modelos.items():
    try:
        clasificador = pipeline("sentiment-analysis", model=modelo)
        for texto_eval in textos_prueba:
            pred = clasificador(texto_eval)[0]
            resultados.append({
                "modelo": nombre,
                "texto": texto_eval,
                "etiqueta": pred["label"],
                "score": pred["score"]
            })
    except Exception as e:
        resultados.append({
            "modelo": nombre,
            "texto": "No ejecutado",
            "etiqueta": "error",
            "score": 0.0
        })
        print(f"No fue posible ejecutar {nombre}: {e}")

df_resultados = pd.DataFrame(resultados)
df_resultados

In [ ]:
# Visualización de scores por modelo
if not df_resultados.empty:
    df_plot = df_resultados[df_resultados["etiqueta"] != "error"].copy()
    if not df_plot.empty:
        df_plot["caso"] = df_plot.groupby("modelo").cumcount() + 1
        plt.figure(figsize=(10, 5))
        for modelo in df_plot["modelo"].unique():
            temp = df_plot[df_plot["modelo"] == modelo]
            plt.plot(temp["caso"], temp["score"], marker="o", label=modelo)
        plt.title("Comparación de confianza por modelo Transformer")
        plt.xlabel("Caso de prueba")
        plt.ylabel("Score")
        plt.ylim(0, 1.05)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No hay resultados válidos para graficar.")

## 6. Complemento opcional: predicción con texto propio

El alumno puede modificar la variable `mi_texto` para probar nuevas entradas.

In [ ]:
# Complemento mínimo: prueba interactiva modificando el texto
mi_texto = "The course was clear, useful and very practical."

try:
    clasificador_final = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )
    print("Texto:", mi_texto)
    print("Predicción:", clasificador_final(mi_texto))
except Exception as e:
    print("No fue posible ejecutar el clasificador final.")
    print("Detalle:", e)